# Financing-availability (coverage) by action — OEF, pre-Mage

**This is a COVERAGE indicator, not fundability** (see `methodology.md` §0). It asks: does a climate-relevant public funding channel of the right sector exist, is it currently usable, and can a city access it? It reflects what we have catalogued (availability bias), not the real-world probability of securing finance. Output: `financing_coverage_by_action.csv` with a `coverage_level` per action (sector-specific / broad-only / none).

In [2]:
import pandas as pd, json
inv=pd.read_csv("data/chile_finance_inventory.csv")
act=pd.read_csv("../../../../climateview/climateview-transition-elements/releases/2026-02-23/data/current_actions.csv")
inv["gpc"]=inv.gpc_sectors.apply(json.loads)
print("funds:",len(inv)," actions:",len(act))
PFX={"I":"stationary_energy","II":"transportation","III":"waste","IV":"industry","V":"afolu"}
def action_sectors(ss):
    s={PFX.get(str(ss).split(".")[0],"cross_sector")}
    if str(ss).startswith("III.4"): s.add("water")
    return s
act["sectors"]=act.subsector_number.apply(action_sectors)
print("action primary-sector spread:", act.sectors.apply(lambda x:sorted(x)[0]).value_counts().to_dict())

funds: 78  actions: 102
action primary-sector spread: {'stationary_energy': 39, 'industry': 28, 'afolu': 17, 'waste': 10, 'transportation': 8}


## Usability + city-access + per-match quality (methodology 2a-2b).

In [3]:
def usable(r):
    s=str(r.status).lower(); rec=str(r.recurrence).lower()
    return ("open" in s) or ("rolling" in s) or rec.startswith("annual") or rec.startswith("ongoing")
inv["usable_now"]=inv.apply(usable,axis=1)
def actor_ok_for_city(r):
    a=str(r.eligible_actor).lower(); ap=str(r.access_pathway).lower()
    if ("not municipal" in a) or ("no municipal" in a): return False   # e.g. "NOT municipalities"
    if "municipal" in a: return True
    if ("facilitated" in ap) or ("via municipality" in ap) or ("municipality applies" in ap): return True
    return False
inv["actor_ok"]=inv.apply(actor_ok_for_city,axis=1)
def match_quality(fund, a_sectors):
    fs=set(fund.gpc); direct=bool((fs-{"cross_sector"}) & a_sectors)
    broad=("cross_sector" in fs) or (fund.specificity=="broad")
    if not (direct or broad): return None
    if fund.detail_level=="index": return "Weak"
    if (fund.specificity=="broad") or (not direct):
        return "Moderate" if (fund.usable_now and fund.actor_ok) else "Weak"
    gaps=(0 if fund.usable_now else 1)+(0 if fund.actor_ok else 1)
    return {0:"Strong",1:"Moderate"}.get(gaps,"Weak")
print("usable funds:",int(inv.usable_now.sum()),"/",len(inv)," | city-accessible:",int(inv.actor_ok.sum()))

usable funds: 49 / 78  | city-accessible: 24


## Classify each action's coverage_level (sector-specific / broad-only / none).

In [4]:
rows=[]
for _,a in act.iterrows():
    q=[(match_quality(f,a.sectors),f.program_name,f.source_dataset) for _,f in inv.iterrows()]
    q=[x for x in q if x[0]]
    nDed=sum(v=="Strong" for v,*_ in q)      # dedicated, usable, city-accessible
    nGen=sum(v=="Moderate" for v,*_ in q)     # general/broad or sector-specific-with-gap
    nWeak=sum(v=="Weak" for v,*_ in q)
    cov="sector-specific" if nDed>=1 else ("broad-only" if nGen>=1 else "none")
    chans=[f"{n} ({d})" for v,n,d in q if v=="Strong"][:3] or [f"{n} ({d})" for v,n,d in q if v=="Moderate"][:3]
    rows.append(dict(action_id=a.action_id,action_name=a.action_name,subsector=a.subsector_number,
        sector=sorted(a.sectors)[0],coverage_level=cov,n_dedicated_channels=nDed,
        n_general_channels=nGen,example_channels="; ".join(chans)))
cov=pd.DataFrame(rows)
cov[["action_name","sector","coverage_level","n_dedicated_channels","n_general_channels"]].head(10)

,action_name,sector,coverage_level,n_dedicated_channels,n_general_channels
0,Introduce energy-efficiency standards for new ...,stationary_energy,sector-specific,2,10
1,Introduce energy-efficiency standards for new ...,stationary_energy,sector-specific,2,10
2,Support Implementation of Industrial Building ...,stationary_energy,sector-specific,2,10
3,Retrofit residential buildings for energy effi...,stationary_energy,sector-specific,2,10
4,Retrofit commercial and institutional other no...,stationary_energy,sector-specific,2,10
5,Retrofit municipal buildings for energy effici...,stationary_energy,sector-specific,2,10
6,Optimize energy efficiency and sustainability ...,stationary_energy,sector-specific,2,10
7,Adopt zero-emission bus fleets for public tran...,transportation,broad-only,0,5
8,Promote deployment of zero-emission freight fl...,transportation,broad-only,0,5
9,Electrify municipal vehicle fleets,transportation,broad-only,0,5


## Coverage summary + integrity checks (broad funds never count as dedicated).

In [5]:
print("coverage_level:"); print(cov.coverage_level.value_counts().to_string())
print(); print("sector x coverage:"); print(cov.groupby(["sector","coverage_level"]).size().unstack(fill_value=0).to_string())
assert not (cov.coverage_level=="sector-specific").all(), "all sector-specific = inflated"
assert cov.coverage_level.nunique()>1, "no spread"
print(); print("broad funds (never dedicated):", sorted(inv[inv.specificity=="broad"].program_name))
print("ASSERTIONS PASSED")

coverage_level:
coverage_level
sector-specific    66
broad-only         36

sector x coverage:
coverage_level     broad-only  sector-specific
sector                                        
afolu                       0               17
industry                   28                0
stationary_energy           0               39
transportation              8                0
waste                       0               10

broad funds (never dedicated): ['FNDR — Inversión Regional (Glosa 03 / SNI)', 'FRIL — Fondo Regional de Iniciativa Local', 'FRPD — Fondo Regional para la Productividad y el Desarrollo', 'Fondo de Recuperación de Ciudades (FRC)', 'Garantías / Coberturas CORFO (e.g. FOGAIN)', 'Innova Chile — I+D e innovación (clean-tech)', 'Pavimentación Participativa', 'Programa de Mejoramiento Urbano y Equipamiento Comunal (PMU)', 'Programa de Mejoramiento de Barrios (PMB)', 'Programa de Recuperación de Barrios (Quiero Mi Barrio)']
ASSERTIONS PASSED


## Export

In [6]:
cov.to_csv("sample/financing_coverage_by_action.csv",index=False)
print("wrote financing_coverage_by_action.csv", cov.shape)

wrote financing_coverage_by_action.csv (102, 8)


## Read-out

- **This is coverage, not fundability** (methodology §0). `sector-specific` = a dedicated, usable, city-accessible channel exists; `broad-only` = only general-purpose funds (PMU/FNDR-type) cover it; `none` = a true gap.
- Coverage is **sector-determined** here (every action in a sector shares a label) — fine for a *where-to-look* map and *gap map*, not for ranking actions.
- Transport & industry are `broad-only` because those sources are unreviewed (availability bias), **not** because such actions are inherently hard to finance.
- For *what actually gets financed in practice*, the next step is a separate **revealed** signal from award/adjudication history — deliberately not built here.